Dataset

   ↓

Train/Test Split

   ↓
   
Untuned Random Forest ──→ Test Performance

   ↓

Training Set only
   ├── GridSearchCV
   └── RandomizedSearchCV

   ↓

 Best Models

  ↓

Same untouched Test Set

  ↓

   Final Comparison

# 🚀 Hyperparameter Optimization

### 🔍 GridSearchCV vs RandomizedSearchCV

> Comparing systematic and randomized hyperparameter search strategies using a machine learning classification model.

In [1]:
# 📦 Step 1 — Import Libraries

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## 📂 Step 2 — Load Dataset

In [2]:
# 📂 Load Telco Customer Churn Dataset

file_path = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(file_path)

print("✅ Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

✅ Dataset loaded successfully!
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 🔍 Step 3 — Dataset Inspection

In [3]:
# 🔍 Inspect Dataset

print("📐 Shape:", df.shape)

print("\n📋 Columns:")
print(df.columns.tolist())

print("\n🧩 Data Types:")
print(df.dtypes)

print("\n❓ Missing Values:")
print(df.isnull().sum())

print("\n🔁 Duplicate Rows:", df.duplicated().sum())

📐 Shape: (7043, 21)

📋 Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

🧩 Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

❓ Missing Values:
customer

## 🧹 Step 4 — Clean Dataset

In [4]:
# 🧹 Clean Data

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())

df = df.dropna(subset=["TotalCharges"]).copy()

print("✅ Shape after cleaning:", df.shape)

Missing TotalCharges: 11
✅ Shape after cleaning: (7032, 21)


## 🎯 Step 5 — Prepare Features & Target

In [5]:
# 🎯 Prepare Features and Target

df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

X = df.drop(columns=["Churn"])
y = df["Churn"]

print("📊 Features:", X.shape)
print("🎯 Target:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

📊 Features: (7032, 19)
🎯 Target: (7032,)

Target Distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


## 🔀 Step 6 — Train/Test Split

In [6]:
# 🔀 Create a Held-Out Test Set

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("🏋️ Training samples:", X_train.shape[0])
print("🧪 Test samples:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

🏋️ Training samples: 5625
🧪 Test samples: 1407

Training target distribution:
Churn
0    4130
1    1495
Name: count, dtype: int64

Test target distribution:
Churn
0    1033
1     374
Name: count, dtype: int64


## 🔢 Step 7 — Encode Features

In [7]:
# 🔢 Encode Categorical Features

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

numeric_features = X_train.select_dtypes(
    exclude=["object"]
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

print("📝 Categorical features:", len(categorical_features))
print("🔢 Numerical features:", len(numeric_features))

📝 Categorical features: 15
🔢 Numerical features: 4


C:\Users\madhu\AppData\Local\Temp\ipykernel_8372\968057508.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


## ⚙️ Step 8 — Transform Training Data

In [8]:
# ⚙️ Fit preprocessing only on training data

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("🏋️ Training shape:", X_train_processed.shape)
print("🧪 Test shape:", X_test_processed.shape)

🏋️ Training shape: (5625, 45)
🧪 Test shape: (1407, 45)


## 🌲 Step 9 — Untuned Baseline

In [9]:
# 🌲 Train Untuned Random Forest Baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(X_train_processed, y_train)

baseline_pred = baseline_model.predict(X_test_processed)

baseline_results = {
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred),
    "Recall": recall_score(y_test, baseline_pred),
    "F1-Score": f1_score(y_test, baseline_pred)
}

print("🌲 Untuned Random Forest Baseline")
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

🌲 Untuned Random Forest Baseline
Accuracy: 0.7925
Precision: 0.6376
Recall: 0.5080
F1-Score: 0.5655


## 🔎 Step 10 — Grid Search Space

In [10]:
# 🔎 Define GridSearchCV Parameter Grid

from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2]
}

total_grid_combinations = (
    len(param_grid["n_estimators"])
    * len(param_grid["max_depth"])
    * len(param_grid["min_samples_split"])
    * len(param_grid["min_samples_leaf"])
)

print("🔢 Total parameter combinations:", total_grid_combinations)
print("🔄 CV folds: 5")
print("📌 Total model fits:", total_grid_combinations * 5)

🔢 Total parameter combinations: 54
🔄 CV folds: 5
📌 Total model fits: 270


## 🔎 Step 11 — GridSearchCV

In [11]:
# 🔎 Run GridSearchCV

grid_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=grid_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_start_time = time.perf_counter()

grid_search.fit(X_train_processed, y_train)

grid_end_time = time.perf_counter()

grid_search_time = grid_end_time - grid_start_time

print("\n✅ Grid Search completed!")
print(f"⏱️ Search time: {grid_search_time:.2f} seconds")
print("🏆 Best CV F1-Score:", f"{grid_search.best_score_:.4f}")
print("⚙️ Best Parameters:")
print(grid_search.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

✅ Grid Search completed!
⏱️ Search time: 58.66 seconds
🏆 Best CV F1-Score: 0.5877
⚙️ Best Parameters:
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300}


## 🎲 Step 12 — Randomized Search

In [12]:
# 🎲 Define RandomizedSearchCV Parameter Space

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(100, 401),
    "max_depth": randint(5, 21),
    "min_samples_split": randint(2, 16),
    "min_samples_leaf": randint(1, 8)
}

print("🎲 Randomized search space created.")
print("🔢 Random iterations:", 54)
print("🔄 CV folds:", 5)
print("📌 Total model fits:", 54 * 5)

🎲 Randomized search space created.
🔢 Random iterations: 54
🔄 CV folds: 5
📌 Total model fits: 270


## 🎲 Step 13 — Run RandomizedSearchCV

In [13]:
# 🎲 Run RandomizedSearchCV

random_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=random_model,
    param_distributions=param_distributions,
    n_iter=54,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_start_time = time.perf_counter()

random_search.fit(X_train_processed, y_train)

random_end_time = time.perf_counter()

random_search_time = random_end_time - random_start_time

print("\n✅ Randomized Search completed!")
print(f"⏱️ Search time: {random_search_time:.2f} seconds")
print("🏆 Best CV F1-Score:", f"{random_search.best_score_:.4f}")
print("⚙️ Best Parameters:")
print(random_search.best_params_)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

✅ Randomized Search completed!
⏱️ Search time: 61.30 seconds
🏆 Best CV F1-Score: 0.5866
⚙️ Best Parameters:
{'max_depth': 9, 'min_samples_leaf': 4, 'min_samples_split': 9, 'n_estimators': 251}


## 🧪 Step 14 — Evaluate Tuned Models

In [14]:
# 🧪 Evaluate Tuned Models on the Unseen Test Set

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

test_results = []

tuned_models = {
    "Baseline": baseline_model,
    "GridSearchCV": grid_search.best_estimator_,
    "RandomizedSearchCV": random_search.best_estimator_
}

for name, model in tuned_models.items():
    y_pred = model.predict(X_test_processed)

    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

test_results_df = pd.DataFrame(test_results)

test_results_df

,Model,Accuracy,Precision,Recall,F1-Score
0,Baseline,0.792466,0.637584,0.508021,0.565476
1,GridSearchCV,0.795309,0.641447,0.521390,0.575221
2,RandomizedSearchCV,0.791045,0.635135,0.502674,0.561194


## 📈 Step 15 — Tuning Improvement

In [15]:
# 📈 Calculate Improvement over Baseline

baseline_f1 = test_results_df.loc[
    test_results_df["Model"] == "Baseline",
    "F1-Score"
].iloc[0]

test_results_df["F1 Improvement vs Baseline"] = (
    test_results_df["F1-Score"] - baseline_f1
)

test_results_df

,Model,Accuracy,Precision,Recall,F1-Score,F1 Improvement vs Baseline
0,Baseline,0.792466,0.637584,0.508021,0.565476,0.000000
1,GridSearchCV,0.795309,0.641447,0.521390,0.575221,0.009745
2,RandomizedSearchCV,0.791045,0.635135,0.502674,0.561194,-0.004282


## ⚖️ Step 16 — Grid vs Randomized Search

In [16]:
# ⚖️ Compare GridSearchCV and RandomizedSearchCV

search_comparison = pd.DataFrame({
    "Search": ["GridSearchCV", "RandomizedSearchCV"],
    "Best CV F1": [
        grid_search.best_score_,
        random_search.best_score_
    ],
    "Search Time (seconds)": [
        grid_search_time,
        random_search_time
    ],
    "Total Fits": [
        270,
        270
    ],
    "Test F1": [
        test_results_df.loc[
            test_results_df["Model"] == "GridSearchCV",
            "F1-Score"
        ].iloc[0],
        test_results_df.loc[
            test_results_df["Model"] == "RandomizedSearchCV",
            "F1-Score"
        ].iloc[0]
    ]
})

search_comparison

,Search,Best CV F1,Search Time (seconds),Total Fits,Test F1
0,GridSearchCV,0.587661,58.657480,270,0.575221
1,RandomizedSearchCV,0.586602,61.303358,270,0.561194


## 🏁 Step 17 — Final Conclusion

In [17]:
# 🏁 Final Conclusion

print("GridSearchCV")
print("-" * 40)
print(f"Best CV F1: {grid_search.best_score_:.4f}")
print(f"Search Time: {grid_search_time:.2f} seconds")
print(f"Test F1: {test_results_df.loc[test_results_df['Model'] == 'GridSearchCV', 'F1-Score'].iloc[0]:.4f}")

print("\nRandomizedSearchCV")
print("-" * 40)
print(f"Best CV F1: {random_search.best_score_:.4f}")
print(f"Search Time: {random_search_time:.2f} seconds")
print(f"Test F1: {test_results_df.loc[test_results_df['Model'] == 'RandomizedSearchCV', 'F1-Score'].iloc[0]:.4f}")

print("\nBaseline")
print("-" * 40)
print(f"Test F1: {baseline_f1:.4f}")

GridSearchCV
----------------------------------------
Best CV F1: 0.5877
Search Time: 58.66 seconds
Test F1: 0.5752

RandomizedSearchCV
----------------------------------------
Best CV F1: 0.5866
Search Time: 61.30 seconds
Test F1: 0.5612

Baseline
----------------------------------------
Test F1: 0.5655


### 💡 Conclusion

GridSearchCV evaluated the complete predefined hyperparameter grid, while RandomizedSearchCV sampled configurations from the defined search distributions using the same 5-fold cross-validation and 270 total fits.

In this experiment, GridSearchCV achieved a slightly higher cross-validation F1-score of 0.5877 compared with 0.5866 from RandomizedSearchCV. On the untouched test set, GridSearchCV achieved an F1-score of 0.5752, compared with 0.5612 for RandomizedSearchCV and 0.5655 for the untuned baseline.

GridSearchCV improved the baseline test F1-score by approximately 0.97 percentage points. RandomizedSearchCV produced a lower test F1-score than the baseline in this particular experiment.

RandomizedSearchCV is generally useful when the hyperparameter space is large or continuous and evaluating every possible combination would be expensive. GridSearchCV is useful when the search space is relatively small and a systematic evaluation of every selected combination is practical.

The final model should therefore be selected based on both validation performance and performance on the untouched test set, while also considering computational cost.